# Z3 (C# / .NET) — Optimisation multi-objectif (lexicographique, Pareto, somme pondérée)

**Navigation** : [Index](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [<< Z3-16e (précédent jumeau C#)](Z3-16e-Meal-Planner-Optimize-CSharp.ipynb) | [Z3-20 — suivant jumeau C# >>](Z3-20-Machine-de-Goedel-CSharp.ipynb)

Twin C# de [Z3-19-Optimization-Lexicographic-Python.ipynb](Z3-19-Optimization-Lexicographic-Python.ipynb) (parité .NET : même instance, mêmes verdicts, documentation en français, API Microsoft.Z3).

Quand un problème d'optimisation a **plusieurs critères contradictoires** (coût vs impact, latence vs débit, performance vs énergie), il n'y a souvent **pas une seule solution optimale** mais un **ensemble de compromis**. Z3 expose `ctx.MkOptimize()`, le solveur dédié à l'optimisation sous contraintes, qui supporte trois familles de stratégies complémentaires :

- **Lexicographique** : on optimise les critères dans un ordre donné (impact d'abord, puis coût, etc.). Une seule   solution sort -- celle qui est meilleure sur le premier critère, et parmi celles-ci la meilleure sur le deuxième, etc.
- **Pareto (front de Pareto)** : on énumère toutes les solutions **non dominées** -- aucune autre solution n'est   strictement meilleure sur tous les critères simultanément. Le résultat est un *ensemble* (souvent petit) qui   représente l'ensemble des compromis rationnels.
- **Somme pondérée** : on agrège les critères en une seule fonction `somme(poids_i * critere_i)`, qu'on maximise   (ou minimise). Le poids de chaque critère reflète l'arbitrage du décideur. Cette méthode est simple mais   **manque certains points du front de Pareto** si la fonction objectif n'est pas concave.

## Problème jouet

On optimise la sélection de **3 projets** sous un **budget de 12 k€** :

| Projet | Coût (k€) | Impact (utilité) |
|--------|----------|-----------------|
| A      | 4        | 6               |
| B      | 5        | 8               |
| C      | 7        | 10              |

Objectifs contradictoires : maximiser l'impact total et minimiser le coût total. Le solveur `MkOptimize()` de Z3 permet d'encoder ces trois stratégies en quelques lignes et de comparer les solutions obtenues.

In [1]:
#r "nuget: Microsoft.Z3"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);

// Culture invariante epinglee : .NET Interactive peut executer chaque soumission sur un
// thread different -- CurrentCulture est par thread, on fixe aussi les defauts du pool.
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

// Contexte Z3 partage par toutes les cellules (meme moteur C++ que z3-solver Python).
var ctx = new Context();
Console.WriteLine($"Moteur : Microsoft.Z3 {Microsoft.Z3.Version.FullVersion}");

// --- Definition du probleme jouet ---
string[] Projets = { "A", "B", "C" };
var couts = new Dictionary<string, int> { ["A"] = 4, ["B"] = 5, ["C"] = 7 };
var impacts = new Dictionary<string, int> { ["A"] = 6, ["B"] = 8, ["C"] = 10 };
const int BUDGET = 12;

// Variables de decision : x[p] = 1 si on selectionne le projet p, 0 sinon
var x = Projets.ToDictionary(p => p, p => ctx.MkIntConst($"x_{p}"));
var opt = ctx.MkOptimize();
foreach (var p in Projets)
{
    opt.Assert(ctx.MkGe(x[p], ctx.MkInt(0)));
    opt.Assert(ctx.MkLe(x[p], ctx.MkInt(1)));
}
opt.Assert(ctx.MkLe(ctx.MkAdd(Projets.Select(p => ctx.MkMul(x[p], ctx.MkInt(couts[p]))).ToArray()), ctx.MkInt(BUDGET)));

ArithExpr coutTotal = ctx.MkAdd(Projets.Select(p => ctx.MkMul(x[p], ctx.MkInt(couts[p]))).ToArray());
ArithExpr impactTotal = ctx.MkAdd(Projets.Select(p => ctx.MkMul(x[p], ctx.MkInt(impacts[p]))).ToArray());

Console.WriteLine("Contraintes posees : chaque projet x_p in {0,1}, somme des couts <= 12 k€");
Console.WriteLine("Objectifs : maximiser impact_total, minimiser cout_total");

Installing Packages Microsoft.Z3

Moteur : Microsoft.Z3 Z3 4.12.2.0


Contraintes posees : chaque projet x_p in {0,1}, somme des couts <= 12 k€


Objectifs : maximiser impact_total, minimiser cout_total


In [2]:
// Mesure 1 -- optimisation lexicographique : maximiser l'impact d'abord, puis minimiser le cout
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

var optLex = ctx.MkOptimize();
foreach (var p in Projets)
{
    optLex.Assert(ctx.MkGe(x[p], ctx.MkInt(0)));
    optLex.Assert(ctx.MkLe(x[p], ctx.MkInt(1)));
}
optLex.Assert(ctx.MkLe(coutTotal, ctx.MkInt(BUDGET)));

// Objectif principal : maximiser impact
optLex.MkMaximize(impactTotal);
// Objectif secondaire : minimiser cout (ne s'active que parmi les solutions d'impact max)
optLex.MkMinimize(coutTotal);

if (optLex.Check() == Status.SATISFIABLE)
{
    var mLex = optLex.Model;
    var selLex = Projets.Where(p => ((IntNum)mLex.Evaluate(x[p])).Int == 1).OrderBy(p => p).ToList();
    int impactVal = ((IntNum)mLex.Evaluate(impactTotal)).Int;
    int coutVal = ((IntNum)mLex.Evaluate(coutTotal)).Int;
    Console.WriteLine($"Lex : selection = [{string.Join(", ", selLex.Select(p => $"\"{p}\""))}] (impact = {impactVal}, cout = {coutVal} k€)");
    Console.WriteLine($"  -> B + C (impact 18, cout 12 k€) : la seule combinaison a impact 18.");
    Console.WriteLine($"  -> Cout secondaire = 12 k€ (le seul faisable pour impact = 18).");
}
else
{
    Console.WriteLine("Lex : aucune solution");
}

Lex : selection = ["B", "C"] (impact = 18, cout = 12 k€)


  -> B + C (impact 18, cout 12 k€) : la seule combinaison a impact 18.


  -> Cout secondaire = 12 k€ (le seul faisable pour impact = 18).


### Interprétation : optimisation lexicographique

La stratégie lex donne une **seule solution** : `B + C` (coût 12 k€, impact 18). Le solveur commence par chercher l'impact maximum faisable (18), et parmi toutes les combinaisons qui atteignent cet impact, il choisit celle de coût minimum. C'est la même solution qu'un décideur rationnel prendrait s'il considère l'impact comme absolument prioritaire.

**Forces** :

- Décision **non-ambiguë** : une seule solution sort, pas d'arbitrage subjectif du décideur.
- Ordre d'importance **explicite** : si A est plus important que B, le solveur respecte cette hiérarchie strictement.
- Le solveur Z3 `ctx.MkOptimize()` implémente cette stratégie nativement via `MkMaximize(...)` puis `MkMinimize(...)`.

**Faiblesses** :

- **Cécité aux autres compromis** : si l'impact maximum est 18 mais qu'une solution à impact 17 coûte 5 k€ de moins,   lex l'ignore. Le décideur ne voit pas le tradeoff.
- **Sensible à l'ordre des critères** : inverser l'ordre (coût d'abord, puis impact) donnerait une solution   différente. C'est cohérent (les critères ne sont pas symétriques), mais cela oblige à un choix d'arbitrage *a priori*.

**Quand l'utiliser** : quand les critères ont une **hiérarchie claire** (e.g. sécurité d'abord, performance ensuite) ou quand le décideur veut une réponse claire et non-discutable.

In [3]:
// Mesure 2 -- enumeration du front de Pareto : on garde les solutions non-dominees
// Strategie : iterer, a chaque etape ajouter une contrainte d'exclusion, jusqu'a plus de solution
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

var solutions = new List<(string[] Sel, int Imp, int Cout)>();
var optP = ctx.MkOptimize();
foreach (var p in Projets)
{
    optP.Assert(ctx.MkGe(x[p], ctx.MkInt(0)));
    optP.Assert(ctx.MkLe(x[p], ctx.MkInt(1)));
}
optP.Assert(ctx.MkLe(coutTotal, ctx.MkInt(BUDGET)));

// On maximise l'impact pour trouver le point extreme 'impact max'
optP.MkMaximize(impactTotal);

while (optP.Check() == Status.SATISFIABLE)
{
    var mP = optP.Model;
    var selP = Projets.Where(p => ((IntNum)mP.Evaluate(x[p])).Int == 1).OrderBy(p => p).ToArray();
    int impP = ((IntNum)mP.Evaluate(impactTotal)).Int;
    int coutP = ((IntNum)mP.Evaluate(coutTotal)).Int;
    solutions.Add((selP, impP, coutP));
    // Contrainte d'exclusion : empecher de retourner exactement a cette solution
    optP.Assert(ctx.MkOr(Projets.Select(p => ctx.MkNot(ctx.MkEq(x[p], mP.Evaluate(x[p])))).ToArray()));
}

Console.WriteLine($"Pareto : {solutions.Count} solutions non-dominnees trouvees\n");
Console.WriteLine($"{"Selection",-12} {"Impact",8} {"Cout (k€)",12}");
Console.WriteLine(new string('-', 36));
foreach (var (sel, imp, cout) in solutions)
{
    string label = sel.Length == 0 ? "aucun" : string.Join("+", sel);
    Console.WriteLine($"{label,-12} {imp,8} {cout,12}");
}

// Filtrer les Pareto-optimales : eliminer les solutions qui en dominent une autre
bool Domine((string[] Sel, int Imp, int Cout) s1, (string[] Sel, int Imp, int Cout) s2)
{
    // s1 domine s2 si s1 a cout <= et impact >=, avec au moins un strictement meilleur.
    return s1.Cout <= s2.Cout && s1.Imp >= s2.Imp && (s1.Cout < s2.Cout || s1.Imp > s2.Imp);
}

var pareto = new List<(string[] Sel, int Imp, int Cout)>();
for (int i = 0; i < solutions.Count; i++)
{
    var s = solutions[i];
    // Exclure la solution vide (ne rien selectionner) : trivialement Pareto-non-dominee
    // (cout 0 minimise le cout, mais impact 0 n'ameliore rien) -- on la filtre pour
    // ne garder que les selections reelles.
    if (s.Sel.Length == 0) continue;
    bool dominated = false;
    for (int j = 0; j < solutions.Count && !dominated; j++)
    {
        if (i == j) continue;
        var t = solutions[j];
        if (t.Sel.Length == 0) continue;
        if (Domine(t, s)) dominated = true;
    }
    if (!dominated) pareto.Add(s);
}
Console.WriteLine($"\nApres filtrage (selection non vide + non-dominance) : {pareto.Count} solutions Pareto-optimales");

Pareto : 7 solutions non-dominnees trouvees



Selection      Impact    Cout (k€)


------------------------------------


B+C                18           12


A+C                16           11


A+B                14            9


C                  10            7


B                   8            5


A                   6            4


aucun               0            0



Apres filtrage (selection non vide + non-dominance) : 6 solutions Pareto-optimales


### Interprétation : front de Pareto

Les **6 solutions** trouvées (B+C, A+C, A+B, C, B, A) sont **toutes Pareto-optimales** : aucune n'est strictement dominée par une autre. Cela vient du fait que les 3 projets ont des **ratios impact/coût différents** (A=1.5, B=1.6, C=1.43) et que chaque solution occupe une niche sur le front.

**Cas dominé vs Pareto-optimal** : on parle de *domination* quand une solution `s1` est meilleure que `s2` sur **tous** les critères simultanément. Ici, par exemple, *aucune* solution n'est meilleure que toutes les autres sur (coût, impact) à la fois -- le problème est multi-objectif authentique, et le front est large.

**Forces** :

- **Visibilité complète des compromis** : le décideur voit *toutes* les options rationnelles (coût min, impact max, intermédiaires).
- **Pas d'arbitrage caché** : le poids de chaque critère n'est pas figé -- le décideur arbitre *après* avoir vu le front.
- **Fondement théorique solide** : la dominance Pareto est l'outil standard en recherche opérationnelle et en économie du bien-être.

**Faiblesses** :

- Le front peut être **trop large** (ici 6 solutions, mais un problème industriel peut en avoir des milliers).
- **Pas de hiérarchie** : le front est un *ensemble*, pas une recommandation -- le décideur doit choisir *dans* le front.
- Coûteux à énumérer : il faut résoudre le problème plusieurs fois (une fois par solution du front).

**Quand l'utiliser** : en phase exploratoire (cadrage d'un nouveau problème), ou quand le décideur veut **voir** le paysage des compromis avant de pondérer.

In [4]:
// Mesure 3 -- somme ponderee : on agrege impact - alpha * cout, et on fait varier alpha
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

// Alphas portes en rationnels exacts (Num/Den) pour l'objectif Z3 (MkReal), avec la
// valeur double associee pour l'affichage -- meme sémantique que le Python (float 1.3).
var alphas = new (double Val, int Num, int Den)[]
{
    (0.0, 0, 1), (0.5, 1, 2), (1.0, 1, 1), (1.3, 13, 10),
    (1.5, 3, 2), (2.0, 2, 1), (5.0, 5, 1), (10.0, 10, 1),
};
var resultats = new List<(double Alpha, string Label, int Imp, int Cout, double Score)>();

foreach (var a in alphas)
{
    var optW = ctx.MkOptimize();
    foreach (var p in Projets)
    {
        optW.Assert(ctx.MkGe(x[p], ctx.MkInt(0)));
        optW.Assert(ctx.MkLe(x[p], ctx.MkInt(1)));
    }
    optW.Assert(ctx.MkLe(coutTotal, ctx.MkInt(BUDGET)));
    // Objectif : maximiser impact - alpha * cout (equivalent a maximiser impact seul si alpha=0,
    // ou minimiser cout seul si alpha est tres grand)
    ArithExpr obj = ctx.MkSub(impactTotal, ctx.MkMul(ctx.MkReal(a.Num, a.Den), coutTotal));
    optW.MkMaximize(obj);
    if (optW.Check() == Status.SATISFIABLE)
    {
        var mW = optW.Model;
        var selW = Projets.Where(p => ((IntNum)mW.Evaluate(x[p])).Int == 1).OrderBy(p => p).ToList();
        int impW = ((IntNum)mW.Evaluate(impactTotal)).Int;
        int coutW = ((IntNum)mW.Evaluate(coutTotal)).Int;
        double scoreW = impW - a.Val * coutW;
        string labelW = selW.Count == 0 ? "aucun" : string.Join("+", selW);
        resultats.Add((a.Val, labelW, impW, coutW, scoreW));
    }
}

Console.WriteLine($"{"alpha",6} {"Selection",-12} {"Impact",8} {"Cout",6} {"Score",8}  Lecture");
Console.WriteLine(new string('-', 70));
foreach (var (alpha, label, imp, cout, score) in resultats)
{
    string lecture = label == "aucun" ? "solution vide (ne rien faire)"
        : alpha == 0.0 ? "max impact seul"
        : "compromis impact / cout";
    Console.WriteLine($"{alpha,6:F2} {label,-12} {imp,8} {cout,5}k {score,8:F2}  {lecture}");
}

// --- Lecture derivee -----------------------------------------------------------
// Les chiffres de la lecture sont CALCULES, pas recopies a la main : on enumere
// toutes les selections faisables, puis, pour chaque alpha, l'ensemble des
// selections optimales. C'est ce qui evite qu'une prose contredise le tableau
// ci-dessus -- et c'est aussi ce qui revele que le front de Pareto contient des
// points que la somme ponderee n'atteint jamais.
List<string[]> Combinaisons(string[] items, int k)
{
    var res = new List<string[]>();
    void Rec(int start, int depth, List<string> cur)
    {
        if (depth == k) { res.Add(cur.ToArray()); return; }
        for (int i = start; i < items.Length; i++)
        {
            cur.Add(items[i]);
            Rec(i + 1, depth + 1, cur);
            cur.RemoveAt(cur.Count - 1);
        }
    }
    Rec(0, 0, new List<string>());
    return res;
}

var FASIBLES = new List<(string[] Sel, int Imp, int Cout)>();
for (int k = 0; k <= Projets.Length; k++)
{
    foreach (var combo in Combinaisons(Projets, k))
    {
        int coutCombo = combo.Sum(p => couts[p]);
        if (coutCombo <= BUDGET)
            FASIBLES.Add((combo, combo.Sum(p => impacts[p]), coutCombo));
    }
}

double ScoreDe((string[] Sel, int Imp, int Cout) s, double alpha) => s.Imp - alpha * s.Cout;

List<(string[] Sel, int Imp, int Cout)> Optimaux(double alpha)
{
    double meilleur = FASIBLES.Max(s => ScoreDe(s, alpha));
    return FASIBLES.Where(s => Math.Abs(ScoreDe(s, alpha) - meilleur) < 1e-9).ToList();
}

string Nom((string[] Sel, int Imp, int Cout) s) => s.Sel.Length == 0 ? "vide" : string.Join("+", s.Sel);

// Python '.4g' : 4 chiffres significatifs, 'inf' pour l'infini (C# rend 'Infinity').
string G4(double v) => double.IsPositiveInfinity(v) ? "inf" : v.ToString("G4", CultureInfo.InvariantCulture);

Console.WriteLine("\nLecture : la solution glisse le long du front de Pareto quand alpha monte.");
foreach (var a in alphas)
{
    var details = string.Join(" | ", Optimaux(a.Val).Select(s => $"{Nom(s)} (impact {s.Imp}, cout {s.Cout} kEUR, score {ScoreDe(s, a.Val):F2})"));
    Console.WriteLine($"  alpha = {a.Val,-5} : {details}");
}

Console.WriteLine("\nSeuils d'alpha deduits des egalites de score :");
Console.WriteLine("  (une selection est optimale exactement sur l'intervalle indique)");
for (int i = 0; i < FASIBLES.Count; i++)
{
    var s = FASIBLES[i];
    double borneInf = 0.0, borneSup = double.PositiveInfinity;
    for (int j = 0; j < FASIBLES.Count; j++)
    {
        if (i == j) continue;  // identite par index : un tuple contenant un tableau ne
        var t = FASIBLES[j];   // supporte pas l'egalite de valeur en C#
        double ecartCout = s.Cout - t.Cout;
        if (ecartCout == 0) continue;
        double alphaEgalite = (double)(s.Imp - t.Imp) / ecartCout;
        if (ecartCout > 0) borneSup = Math.Min(borneSup, alphaEgalite);
        else borneInf = Math.Max(borneInf, alphaEgalite);
    }
    if (borneInf <= borneSup)
        Console.WriteLine($"  {Nom(s),-8} : alpha dans [{G4(borneInf)}, {G4(borneSup)}]");
}

 alpha Selection      Impact   Cout    Score  Lecture


----------------------------------------------------------------------


  0.00 B+C                18    12k    18.00  max impact seul


  0.50 B+C                18    12k    12.00  compromis impact / cout


  1.00 B+C                18    12k     6.00  compromis impact / cout


  1.30 B+C                18    12k     2.40  compromis impact / cout


... (15 outputs left out)

  vide     : alpha dans [1.6, inf]


  B        : alpha dans [1.5, 1.6]


  A+B      : alpha dans [1.333, 1.5]


  B+C      : alpha dans [0, 1.333]


### Interprétation : somme pondérée

La somme pondérée `maximiser(impact - α * coût)` fait varier la solution selon `α` :

- `α = 0` : on ignore le coût, on maximise l'impact → **B+C** (impact 18).
- `α = 0.5` à `α = 1.0` : compromis équilibré → **B+C** (le score 18 - α × 12 reste le plus élevé).
- `α = 1.3` : le coût commence à peser, mais pas encore assez pour changer de solution → **B+C** (score 2.4),   devant **A+B** (score 2.3). Les deux s'égalisent exactement à **α = 4/3 ≈ 1.33** : c'est le seuil où B+C   cède la place à A+B.
- `α = 1.5` : **A+B** et **B** sont les deux solutions optimales, à score 0.5 chacune (le solveur n'en retourne   qu'une). A, B+C et la solution vide sont à 0 ; A+C et C sont à -0.5. Le coût pèse alors assez pour que le   projet le plus rentable (B) concurrence le couple A+B.
- `α > 8/5 = 1.6` : la **solution vide** devient optimale (coût 0, impact 0, score 0). À `α = 1.6` elle est à   égalité avec B, et pour `α ≥ 2` tous les scores non vides sont strictement négatifs. Le modèle dit   littéralement *"ne rien sélectionner"*.
- `α = 10` : la solution vide reste optimale (coût >> impact, plus aucun projet ne vaut son coût).

**Forces** :

- **Simple** : un seul paramètre à régler (le vecteur de poids), une seule résolution par jeu de poids.
- **Continuité** : faire varier `α` continûment fait *glisser* la solution le long du front de Pareto.
- **Sensibilité** : on peut étudier comment la décision réagit à un changement d'arbitrage du décideur.

**Faiblesses** :

- **Pas concave → trous** : la somme pondérée ne parcourt que l'**enveloppe concave supérieure** du front de   Pareto. Ici, parmi les 6 solutions Pareto-optimales, seules **B+C**, **A+B** et **B** ont un intervalle d'α où   elles sont optimales ; **A, C et A+C** sont Pareto-optimales mais **jamais choisies** par la somme pondérée   (le « trou » typique des points non supportés). Un front concave serait, lui, entièrement exploré — ce n'est   pas garanti en général.
- **Solution vide possible** : si le poids du coût devient trop fort, la solution vide (ne rien faire) devient   optimale (ici au-delà de `α = 8/5`). C'est cohérent mathématiquement, mais c'est un signal d'alarme : le décideur   veut sans doute dépenser le budget, et la modélisation doit intégrer cette intention (par exemple via une   contrainte `somme(x_p) >= 1`).
- **Interprétabilité du poids** : dire "α = 2" n'a pas de signification concrète pour un décideur non technique.
- **Tirage au sort des poids** : pour échantillonner le front de Pareto, on tire `α` au hasard -- certains points   du front sont sur-représentés (concavité), d'autres sous-représentés.

**Quand l'utiliser** : quand le décideur peut exprimer un arbitrage numérique (même grossier) entre critères, ou pour faire une **analyse de sensibilité** (que se passe-t-il si je change d'avis ?). **Attention** : ajouter une contrainte `au moins 1 projet` si l'objectif du décideur est de dépenser le budget.

In [5]:
// Exercice 1 -- Optimisation lexicographique sur 4 projets
//
// Probleme : 4 projets D, E, F, G avec les caracteristiques suivantes :
//   D : cout 3, impact 5
//   E : cout 6, impact 9
//   F : cout 4, impact 7
//   G : cout 5, impact 8
// Budget total : 14 k€.
//
// Coder une optimisation lexicographique qui maximise l'impact d'abord, puis minimise le cout.
// Affichez la selection finale et l'impact/cout associes.

void ExerciceLex4Vars()
{
    // A completer : coder la strategie lex pour 4 projets.
    // TODO etudiant : creer l'instance ctx.MkOptimize(), poser les contraintes 0 <= x_p <= 1,
    // ajouter la contrainte budget, puis MkMaximize(impact) puis MkMinimize(cout).
    // Afficher la selection finale.
}

ExerciceLex4Vars();
Console.WriteLine("Exercice 1 : voir cellule ci-dessus. Selection attendue = ? (a calculer)");

Exercice 1 : voir cellule ci-dessus. Selection attendue = ? (a calculer)


In [6]:
// Exercice 2 -- Enumeration du front de Pareto sur 4 projets
//
// Reprenez les memes 4 projets D, E, F, G que l'exercice 1.
// Enumerez TOUTES les solutions Pareto-optimales (selection, impact, cout).
// Affichez le nombre de solutions et le tableau resultat.

void ExercicePareto4Vars()
{
    // A completer : enumerer le front de Pareto pour 4 projets.
    // TODO etudiant : reprendre la strategie d'enumeration de la mesure 2 (cellule 4),
    // l'adapter aux 4 projets D/E/F/G avec leur budget 14 k€.
}

ExercicePareto4Vars();
Console.WriteLine("Exercice 2 : voir cellule ci-dessus. Nombre de solutions Pareto = ? (a calculer)");

Exercice 2 : voir cellule ci-dessus. Nombre de solutions Pareto = ? (a calculer)


In [7]:
// Exercice 3 -- Somme ponderee sur 4 projets avec 4 poids differents
//
// Reprenez les 4 projets D, E, F, G. Testez 4 valeurs de alpha (par exemple 0, 1, 2, 5).
// Pour chaque alpha, affichez la selection choisie par le solveur.
// Concluez : pour quelle valeur de alpha la selection change-t-elle ?

void ExerciceWeighted4Poids()
{
    // A completer : faire varier alpha et observer la selection optimale.
    // TODO etudiant : boucle sur 4 valeurs de alpha, appel a ctx.MkOptimize() a chaque fois,
    // afficher (alpha, selection, impact, cout).
}

ExerciceWeighted4Poids();
Console.WriteLine("Exercice 3 : voir cellule ci-dessus. Verdict attendu = \"alpha=0 -> D+E+G (impact 22) ; " +
    "alpha=1 -> D+E+G, ex aequo avec D+E+F et D+F+G (score 8) ; alpha=2 et alpha=5 -> solution vide " +
    "(aucun projet, score 0)\" ou similaire.");

Exercice 3 : voir cellule ci-dessus. Verdict attendu = "alpha=0 -> D+E+G (impact 22) ; alpha=1 -> D+E+G, ex aequo avec D+E+F et D+F+G (score 8) ; alpha=2 et alpha=5 -> solution vide (aucun projet, score 0)" ou similaire.


## Conclusion : arbre de décision pour choisir sa stratégie

Le tableau ci-dessous résume quand utiliser lex, Pareto ou weighted, en fonction du contexte du décideur.

| Situation du décideur | Stratégie recommandée | Justification |
|----------------------|----------------------|---------------|
| **Hiérarchie claire des critères** (e.g. sécurité > coût > performance) | **Lexicographique** | L'ordre d'importance est explicite ; une seule solution sort, pas d'arbitrage subjectif |
| **Cadrage exploratoire** (premier contact avec un nouveau problème) | **Pareto** | Le front complet révèle la structure des compromis ; le décideur voit *avant* de pondérer |
| **Analyse de sensibilité** (que se passe-t-il si je change d'avis ?) | **Somme pondérée avec sweep sur α** | On observe comment la solution glisse le long du front ; utile pour le décideur hésitant |
| **Décideur non technique, arbitrage grossier** | **Somme pondérée** | Facile à expliquer ("je préfère 2 fois plus le coût à l'impact") |
| **Front non-concave** (cas pathologique) | **Pareto** | La somme pondérée ne peut pas atteindre les points non-concaves ; seul Pareto les capture |

**Take-away** : les trois stratégies sont **complémentaires**, pas concurrentes. Dans la pratique, on commence par Pareto pour *voir* le paysage, on teste une ou deux valeurs de `α` pour sentir la *sensibilité*, et on finit par lex pour *trancher* quand le décideur a une hiérarchie claire en tête.

**Pour aller plus loin** :

- **Z3-06 Advanced Optimization (jumeau C#)** : explore d'autres facettes (soft constraints, weighted MaxSAT).
- **Z3-12 Arithmétique réelle (jumeau C#)** : si les variables sont réelles (pas seulement entières 0/1), les solutions   deviennent continues et le front est un *polyèdre* dans l'espace des objectifs.
- **Documentation Z3 `Optimize`** : https://microsoft.github.io/z3guide/programming/Z3%20JavaScript%20Examples/optimization.html